In [1]:
!pip install cvxpy control --quiet
print('OK')

OK



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install casadi


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
"""
Koopman Observer v12 — PARÁMETROS PLANTA REAL
===============================================
Actualizado desde la versión de simulación (benchmark TU Wien) a los
parámetros calibrados experimentalmente de la planta física construida:
  - Tanques cilíndricos PVC: Atank = 3.02e-3 m² (D = 6.2 cm)
  - Orificios calibrados por vaciado libre: αo = 0.0271, Do = 10 mm
  - Bombas R385 limitadas al 70%: qi1max=2.69e-5, qi3max=2.41e-5 m³/s
  - Referencias físicamente alcanzables: 0.10 → 0.20 → 0.15 m
  - Ruido sensor HC-SR04: σ = 1 mm (resolución real del sensor)

CAMBIOS respecto a la versión benchmark:
  [P1] Todos los parámetros físicos actualizados a planta real calibrada
  [P2] Referencias cambiadas de 0.10→0.25→0.15 a 0.10→0.20→0.15 m
  [P3] σ = 1e-3 m (HC-SR04 real) en lugar de 2e-3 m (benchmark)
  [P4] SS calculados para h2 ∈ {0.10, 0.15, 0.20} (rango planta real)
  [P5] Bloques EDMD adaptados al rango operativo real [0.05, 0.22] m

Mantiene sin cambio:
  - Diccionario de 8 observables: [√h1,√h2,√h3, h1h2,h2h3,h1h3, h1,h3]
  - Matriz C de 4 filas: [√h1, √h3, h1, h3]
  - DARE calibrada con Q = α·cov(residuos_EDMD)
  - Estructura del observador Luenberger
"""

import numpy as np
import matplotlib.pyplot as plt
import control
from scipy.optimize import least_squares
import scipy.linalg as la
import os

# ============================================================================
# [P1] PARÁMETROS FÍSICOS — PLANTA REAL CALIBRADA
# ============================================================================
p = {
    'Atank'   : 3.02e-3,
    'rho'     : 997.0,        'eta'      : 8.9e-4,   'g'        : 9.81,
    'alphao1' : 0.0271,  'Do1': 10e-3,
    'alphao2' : 0.0271,  'A2' : 7.85e-5,
    'alphao3' : 0.0271,  'Do3': 10e-3,
    'alpha120': 0.3038,  'D12': 10e-3,  'A12': 7.85e-5,  'lambdac12': 24000,
    'alpha230': 0.1344,  'D23': 10e-3,  'A23': 7.85e-5,  'lambdac23': 29600,
    'qi1max'  : 2.69e-5,
    'qi3max'  : 2.41e-5,
}

Ts    = 2.0; T_sim = 800; Nsim  = int(T_sim / Ts)
sigma = 1e-3; EPS = 1e-10; EPS_PSI = 1e-8

# ============================================================================
# MODELO NO LINEAL
# ============================================================================
def nl_ode(x, u):
    h1 = max(x[0], EPS); h2 = max(x[1], EPS); h3 = max(x[2], EPS)
    qi1 = p['qi1max'] * np.clip(u[0], 0, 1)
    qi3 = p['qi3max'] * np.clip(u[1], 0, 1)
    qo1 = p['alphao1'] * (p['Do1']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h1)
    qo2 = p['alphao2'] * p['A2']                    * np.sqrt(2 * p['g'] * h2)
    qo3 = p['alphao3'] * (p['Do3']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h3)
    dh12 = h1 - h2
    l12  = p['D12'] * p['rho'] / p['eta'] * np.sqrt(2 * p['g'] * abs(dh12) + EPS)
    a12  = p['alpha120'] * np.tanh(2 * l12 / p['lambdac12'])
    q12  = a12 * p['A12'] * np.sqrt(2 * p['g'] * abs(dh12) + EPS) * np.sign(dh12 + 1e-15)
    dh23 = h2 - h3
    l23  = p['D23'] * p['rho'] / p['eta'] * np.sqrt(2 * p['g'] * abs(dh23) + EPS)
    a23  = p['alpha230'] * np.tanh(2 * l23 / p['lambdac23'])
    q23  = a23 * p['A23'] * np.sqrt(2 * p['g'] * abs(dh23) + EPS) * np.sign(dh23 + 1e-15)
    return np.array([
        (qi1 - q12 - qo1) / p['Atank'],
        (q12 - q23 - qo2) / p['Atank'],
        (qi3 + q23 - qo3) / p['Atank'],
    ])

def rk4(x, u, dt=Ts):
    k1 = nl_ode(x, u); k2 = nl_ode(x + dt/2 * k1, u)
    k3 = nl_ode(x + dt/2 * k2, u); k4 = nl_ode(x + dt  * k3, u)
    return np.clip(x + dt/6 * (k1 + 2*k2 + 2*k3 + k4), 0, 0.25)

def psi(x):
    h1, h2, h3 = x[0], x[1], x[2]
    return np.array([
        np.sqrt(max(h1, EPS_PSI)), np.sqrt(max(h2, EPS_PSI)), np.sqrt(max(h3, EPS_PSI)),
        h1 * h2, h2 * h3, h1 * h3, h1, h3,
    ], dtype=float)

# ============================================================================
# ESTADOS ESTACIONARIOS
# ============================================================================
def compute_ss(h2_ref):
    def eqs(v):
        h1, h3, u1 = v
        return nl_ode([max(h1, 1e-4), h2_ref, max(h3, 1e-4)],
                      [np.clip(u1, 0, 1), 0.0])
    guesses = [
        (h2_ref * 1.4, h2_ref * 0.8, 0.5), (h2_ref * 1.6, h2_ref * 0.7, 0.7),
        (h2_ref * 2.0, h2_ref * 0.8, 0.9), (h2_ref * 2.5, h2_ref * 0.8, 0.7),
        (h2_ref * 2.5, h2_ref * 0.8, 0.75),(h2_ref * 1.8, h2_ref * 0.85, 0.85),
        (h2_ref * 1.2, h2_ref * 0.9, 0.4), (h2_ref * 1.3, h2_ref * 0.75, 0.6),
    ]
    best, best_res = None, 1e10
    for g in guesses:
        try:
            s = least_squares(eqs, list(g),
                              bounds=([0.01, 0.01, 0.0], [0.25, 0.25, 1.0]),
                              max_nfev=5000)
            if s.success and np.linalg.norm(s.fun) < best_res:
                best_res = np.linalg.norm(s.fun); best = s.x
        except: pass
    if best is not None and best_res < 1e-4:
        return np.array([best[0], h2_ref, best[1]]), np.array([np.clip(best[2], 0, 1), 0.0])
    return None, None

print("Calculando estados estacionarios (parámetros planta real)...")
SS = {}
for h2r in [0.10, 0.15, 0.20]:
    x_ss, u_ss = compute_ss(h2r)
    if x_ss is not None:
        SS[h2r] = (x_ss, u_ss)
        print(f"  h2={h2r:.2f}m → x_ss={np.round(x_ss, 4)}, u_ss={np.round(u_ss, 4)}")
    else:
        print(f"  h2={h2r:.2f}m → SS NO encontrado")

print(f"\nEDMD — 3 bloques, nz=8, lam=1e-4")

# ============================================================================
# RECOLECCIÓN DE DATOS EDMD
# ============================================================================
def collect_data(seed=42):
    rng = np.random.default_rng(seed)
    z_k, u_k, z_k1 = [], [], []

    for h2r in [0.08, 0.10, 0.12, 0.15, 0.17, 0.20, 0.22]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None: continue
        for pert in [0.95, 0.90, 0.98, 0.85, 0.92, 0.88]:
            x = x_ss.copy(); x[1] = np.clip(h2r * pert, 0.04, 0.22)
            for _ in range(170):
                err = x[1] - h2r; u = u_ss.copy()
                u[0] += np.clip(-0.35 * err, -0.35, 0.35) + rng.uniform(-0.15, 0.15)
                u[1] += rng.uniform(-0.05, 0.05); u = np.clip(u, 0, 1)
                z = psi(x); xn = rk4(x, u, Ts); zn = psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None: continue
        for pert in [0.60, 0.70, 0.75, 1.20, 1.30]:
            x = x_ss.copy(); x[1] = np.clip(h2r * pert, 0.04, 0.22)
            for _ in range(100):
                err = x[1] - h2r; u = u_ss.copy()
                u[0] += np.clip(-0.6 * err, -0.6, 0.6) + rng.uniform(-0.1, 0.1)
                u = np.clip(u, 0, 1)
                z = psi(x); xn = rk4(x, u, Ts); zn = psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    for (h2f, h2t) in [(0.10, 0.20), (0.20, 0.15), (0.15, 0.10)]:
        xf, uf = compute_ss(h2f); xt, _ = compute_ss(h2t)
        if xf is None or xt is None: continue
        x = xf.copy()
        for _ in range(200):
            err = x[1] - h2t; u = uf.copy()
            u[0] += np.clip(-0.5 * err, -0.5, 0.5) + rng.uniform(-0.08, 0.08)
            u = np.clip(u, 0, 1)
            z = psi(x); xn = rk4(x, u, Ts); zn = psi(xn)
            z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    return np.array(z_k), np.array(u_k), np.array(z_k1)

Z, U, Zn = collect_data(seed=42)
nz, nu = Z.shape[1], U.shape[1]
print(f"  {len(Z)} muestras recolectadas, nz={nz}")

Phi = np.hstack([Z, U]); lam = 1e-4
Theta = np.linalg.solve(Phi.T @ Phi + lam * np.eye(nz + nu), Phi.T @ Zn)
A_raw = Theta[:nz, :].T; B_d = Theta[nz:, :].T

vals, vecs = np.linalg.eig(A_raw); delta = 1e-4
vals_s = np.where(np.abs(vals) >= 1.0, vals / np.abs(vals) * (1.0 - delta), vals)
A_d = np.real(vecs @ np.diag(vals_s) @ np.linalg.inv(vecs))

rmse_id  = np.sqrt(np.mean((Z @ A_d.T + U @ B_d.T - Zn)**2))
eigs_d   = np.linalg.eigvals(A_d)
print(f"  RMSE one-step: {rmse_id:.2e}")
print(f"  |λ(A_d)| ∈ [{np.abs(eigs_d).min():.5f}, {np.abs(eigs_d).max():.5f}]  "
      f"Estable: {np.abs(eigs_d).max() < 1} ✓")

# ============================================================================
# DISEÑO DEL OBSERVADOR
# ============================================================================
m = 4
C = np.zeros((m, nz))
C[0, 0] = 1.0; C[1, 2] = 1.0; C[2, 6] = 1.0; C[3, 7] = 1.0

O_mat = control.obsv(A_d, C)
rank  = np.linalg.matrix_rank(O_mat, tol=1e-8)
print(f"\n  Observabilidad (C 4×8): {rank}/{nz} {'✓' if rank == nz else '✗'}")

print("\n--- Diseño de ganancia L (DARE con Q = Q_OBSERVER·I, igual que v3) ---")
Q_OBSERVER = 0.1
L_d = None; fuente = ''

for q_try in [Q_OBSERVER, 0.05, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]:
    try:
        Q_dare = q_try * np.eye(nz)
        R_obs  = np.eye(m)
        X      = la.solve_discrete_are(A_d.T, C.T, Q_dare, R_obs)
        L_c    = np.linalg.solve(C @ X @ C.T + R_obs, C @ X @ A_d.T).T
        eigs   = np.linalg.eigvals(A_d - L_c @ C)
        max_mod = np.abs(eigs).max()
        max_L   = np.abs(L_c).max()
        print(f"  q={q_try}: max|λ|={max_mod:.5f}  max|L|={max_L:.4f}", end='')
        if max_mod < 1.0 and max_L > 0.01 and max_L < 500:
            L_d = L_c; fuente = f'DARE q={q_try}·I'
            print("  ✓ ACEPTADO"); break
        else:
            print("  descartado")
    except Exception as e:
        print(f"  error: {e}")

if L_d is None:
    raise RuntimeError("No se pudo diseñar L.")

eigs_final = np.linalg.eigvals(A_d - L_d @ C)
print(f"\nGanancia L: {fuente}  |  max|L|={np.abs(L_d).max():.4f}")
print(f"  Polo dominante: |λ|_max = {np.abs(eigs_final).max():.5f}")

# ============================================================================
# SIMULACIÓN DEL OBSERVADOR
# ============================================================================
def simulate(x0_true, x0_hat, u_func, T_sim=800, seed=0):
    rng_loc = np.random.default_rng(seed)
    N   = int(T_sim / Ts)
    t_arr   = np.arange(N + 1) * Ts
    xt_hist = np.zeros((3, N + 1)); xh_hist = np.zeros((3, N + 1))
    err_arr = np.zeros(N + 1)

    x = x0_true.copy(); z = psi(x0_hat)
    xt_hist[:, 0] = x; xh_hist[:, 0] = np.clip(z[:3]**2, 0, 0.25)
    err_arr[0] = np.linalg.norm(x - xh_hist[:, 0])

    for k in range(N):
        u = u_func(k)
        z_pred = A_d @ z + B_d @ u
        z_pred[:3] = np.maximum(z_pred[:3], 0)
        z_pred[6:] = np.maximum(z_pred[6:], 0)

        xn = rk4(x, u, Ts)
        y_raw_h1 = max(xn[0] + sigma * rng_loc.standard_normal(), 0)
        y_raw_h3 = max(xn[2] + sigma * rng_loc.standard_normal(), 0)
        y_obs = np.array([
            np.sqrt(max(y_raw_h1, EPS_PSI)), np.sqrt(max(y_raw_h3, EPS_PSI)),
            y_raw_h1, y_raw_h3,
        ])
        innov = y_obs - C @ z_pred
        z     = z_pred + L_d @ innov
        z[:3] = np.maximum(z[:3], 0); z[6:] = np.maximum(z[6:], 0)

        x_hat = np.clip(z[:3]**2, 0, 0.25)
        x = xn; xt_hist[:, k+1] = x; xh_hist[:, k+1] = x_hat
        err_arr[k+1] = np.linalg.norm(x - x_hat)

    return t_arr, xt_hist, xh_hist, err_arr

t1 = int(Nsim * 0.33); t2 = int(Nsim * 0.66)
yref_v = np.concatenate([
    0.10 * np.ones(t1), 0.20 * np.ones(t2 - t1), 0.15 * np.ones(Nsim - t2)
])
ref_change_times_s = [t1 * Ts, t2 * Ts]

x_ss_init, u_ss_init = SS.get(0.10, (None, None))
if x_ss_init is None:
    raise RuntimeError("SS para h2=0.10 no encontrado.")

def u_schedule(k):
    h2r = yref_v[min(k, Nsim - 1)]
    xs, us = SS.get(h2r, (None, None))
    if us is None: xs, us = compute_ss(h2r)
    return us if us is not None else u_ss_init

x0_true = x_ss_init + np.array([0.005, -0.008, 0.004])
x0_hat  = x_ss_init.copy()

print(f"\nSimulando observador (parámetros planta real)...")
print(f"  x0_true = {np.round(x0_true, 4)}")
print(f"  Ts={Ts}s | T_sim={T_sim}s | σ={sigma*1000:.0f}mm")

t_arr, xt, xh, err = simulate(x0_true, x0_hat, u_schedule, T_sim=T_sim, seed=42)

# ============================================================================
# MÉTRICAS
# ============================================================================
SETTLING_S = 60
ref_full = np.append(yref_v, yref_v[-1])
mask_400 = t_arr > 400
settling_mask = np.ones(len(t_arr), dtype=bool)
for tc in ref_change_times_s:
    settling_mask &= ~((t_arr >= tc) & (t_arr < tc + SETTLING_S))
mask_ss = mask_400 & settling_mask

e1 = (xt[0, :] - xh[0, :])[mask_ss]
e2 = (xt[1, :] - xh[1, :])[mask_ss]
e3 = (xt[2, :] - xh[2, :])[mask_ss]
rmse_h1 = np.sqrt(np.mean(e1**2)) * 100
rmse_h2 = np.sqrt(np.mean(e2**2)) * 100
rmse_h3 = np.sqrt(np.mean(e3**2)) * 100

mhe_rmse = {'h1': 0.0339, 'h2': 0.0291, 'h3': 0.0357}  # de mhe_standalone.ipynb

print(f"\n{'═'*60}")
print(f"  RMSE Observador Koopman — Simulación Planta Real (SS, excl. {SETTLING_S}s)")
print(f"{'═'*60}")
print(f"  h1 (medido)     = {rmse_h1:.4f} cm")
print(f"  h2 (NO medido)  = {rmse_h2:.4f} cm  ← variable de interés")
print(f"  h3 (medido)     = {rmse_h3:.4f} cm")

# ============================================================================
# GRÁFICAS — estilo unificado con Koopman Closed-Loop v3
# ============================================================================
e1f = (xt[0] - xh[0]) * 100
e2f = (xt[1] - xh[1]) * 100
e3f = (xt[2] - xh[2]) * 100

fig, axes = plt.subplots(3, 2, figsize=(13, 11))
fig.suptitle(
    f'Observador Koopman — Simulación\n'
    f'RMSE_SS (excl. trans.): h₁={rmse_h1:.3f} h₂={rmse_h2:.3f} h₃={rmse_h3:.3f} cm',
    fontsize=9, fontweight='bold')

# ── Panel 1: h1 ─────────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(t_arr, xt[0]*100, 'b', lw=1.5, label='$h_1$ real')
ax.plot(t_arr, xh[0]*100, 'r--', lw=1.5, label='$h_1$ estimado')
ax.set_ylabel('$h_1$ [cm]'); ax.set_xlabel('t [s]')
ax.set_title('Tanque 1'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 2: h2 con referencia ──────────────────────────────────────────────
ax = axes[0, 1]
ax.plot(t_arr, xt[1]*100, 'b', lw=2, label='$h_2$ real')
ax.plot(t_arr, xh[1]*100, 'r--', lw=2, label='$h_2$ estimado (Koopman)')
ax.stairs(ref_full[:-1]*100, t_arr, color='k', linestyle='--', lw=1.5, label='Referencia')
ax.set_ylabel('$h_2$ [cm]'); ax.set_xlabel('t [s]')
ax.set_title('Tanque 2 — NO medido'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 3: h3 ─────────────────────────────────────────────────────────────
ax = axes[1, 0]
ax.plot(t_arr, xt[2]*100, 'b', lw=1.5, label='$h_3$ real')
ax.plot(t_arr, xh[2]*100, 'r--', lw=1.5, label='$h_3$ estimado')
ax.set_ylabel('$h_3$ [cm]'); ax.set_xlabel('t [s]')
ax.set_title('Tanque 3'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 4: error de estimación ────────────────────────────────────────────
ax = axes[1, 1]
ax.plot(t_arr, e1f, 'b', lw=1.2, label=f'$e_{{h1}}$ RMSE_SS={rmse_h1:.3f} cm')
ax.plot(t_arr, e2f, 'r', lw=1.5, label=f'$e_{{h2}}$ RMSE_SS={rmse_h2:.3f} cm (NO medido)')
ax.plot(t_arr, e3f, color=[0, .6, 0], lw=1.2, label=f'$e_{{h3}}$ RMSE_SS={rmse_h3:.3f} cm')
ax.axhline(0, color='k', ls=':')
ax.axvline(400, color='gray', lw=0.8, ls=':', label='t=400s')
for i, tc in enumerate(ref_change_times_s):
    ax.axvspan(tc, tc + SETTLING_S, alpha=0.12, color='orange',
               label='transitorio excluido' if i == 0 else None)
ax.set_ylabel('Error [cm]'); ax.set_xlabel('t [s]')
ax.set_title('Error de estimación')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 5: decaimiento del error ──────────────────────────────────────────
ax = axes[2, 0]
ax.semilogy(t_arr, np.maximum(err, 1e-6)*100, 'b', lw=1.5)
ax.axvline(400, color='gray', lw=0.8, ls=':', label='t=400s')
ax.set_ylabel(r'$\|e\|_2$ [cm]'); ax.set_xlabel('t [s]')
ax.set_title('Decaimiento del error de estimación')
ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.3)

# ── Panel 6: barras comparativas RMSE ───────────────────────────────────────
ax = axes[2, 1]
xp = np.arange(3); w = 0.35
if any(v > 0 for v in mhe_rmse.values()):
    ax.bar(xp - w/2, [mhe_rmse['h1'], mhe_rmse['h2'], mhe_rmse['h3']],
           w, label='MHE standalone', color='steelblue', alpha=0.82)
    ax.bar(xp + w/2, [rmse_h1, rmse_h2, rmse_h3],
           w, label='Koopman Observer', color='forestgreen', alpha=0.82)
    ax.legend(fontsize=8)
else:
    ax.bar(xp, [rmse_h1, rmse_h2, rmse_h3], color='forestgreen', alpha=0.82)
    ax.text(0.5, 0.5, 'Actualizar mhe_rmse\ntras correr MHE standalone',
            ha='center', va='center', transform=ax.transAxes, fontsize=9, color='red')
ax.set_xticks(xp)
ax.set_xticklabels(['$h_1$', '$h_2$ (NO medida)', '$h_3$'])
ax.set_ylabel('RMSE_SS [cm]')
ax.set_title('Comparativa RMSE_SS')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
output_dir = './outputs_sim'
os.makedirs(output_dir, exist_ok=True)
fname = os.path.join(output_dir, 'koopman_observer_planta_real.png')
plt.savefig(fname, dpi=200, bbox_inches='tight'); plt.close()
print(f"\nFigura guardada: {fname}")

Calculando estados estacionarios (parámetros planta real)...
  h2=0.10m → x_ss=[0.114  0.1    0.0816], u_ss=[0.3293 0.    ]
  h2=0.15m → x_ss=[0.1676 0.15   0.1267], u_ss=[0.4039 0.    ]
  h2=0.20m → x_ss=[0.2207 0.2    0.1725], u_ss=[0.4664 0.    ]

EDMD — 3 bloques, nz=8, lam=1e-4
  8220 muestras recolectadas, nz=8
  RMSE one-step: 1.03e-04
  |λ(A_d)| ∈ [0.03515, 0.99990]  Estable: True ✓

  Observabilidad (C 4×8): 8/8 ✓

--- Diseño de ganancia L (DARE con Q = Q_OBSERVER·I, igual que v3) ---
  q=0.1: max|λ|=0.95038  max|L|=0.2250  ✓ ACEPTADO

Ganancia L: DARE q=0.1·I  |  max|L|=0.2250
  Polo dominante: |λ|_max = 0.95038

Simulando observador (parámetros planta real)...
  x0_true = [0.119  0.092  0.0856]
  Ts=2.0s | T_sim=800s | σ=1mm

════════════════════════════════════════════════════════════
  RMSE Observador Koopman — Simulación Planta Real (SS, excl. 60s)
════════════════════════════════════════════════════════════
  h1 (medido)     = 0.0322 cm
  h2 (NO medido)  = 0.0273 cm  ← v